# Anggota 4 — Ekstraksi Jawaban

Rimba Nevada · 245150201111013

# 1. Aturan Simbolik

In [1]:
import json
import operator
import re
from pathlib import Path

import pandas as pd
import spacy_udpipe

_UNIT_BILANGAN = (
    "sks|semester|menit|jam|hari|minggu|bulan|tahun|kali|orang|publikasi|jenis"
)


_DASAR = {
    "nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4,
    "lima": 5, "enam": 6, "tujuh": 7, "delapan": 8, "sembilan": 9,
}


_KOMPARATOR = {
    "<": operator.lt,
    "≤": operator.le,
    "<=": operator.le,
    ">": operator.gt,
    "≥": operator.ge,
    ">=": operator.ge,
}


_POLA_KURUNG_ANGKA = re.compile(r"\b(\d+(?:,\d+)?)\s*\([a-z\s-]+\)", re.IGNORECASE)


_POLA_KATA_BILANGAN = re.compile(
    rf"\b(?!(?:salah\s+)?satu\b(?!\s+(?:{_UNIT_BILANGAN})\b))"
    r"(?:(?:seratus|(?:" + "|".join(_DASAR) + r")\s+ratus)\s+)?"
    r"(?:(?:sepuluh|sebelas|(?:" + "|".join(_DASAR) + r")\s+belas)|"
    r"(?:(?:" + "|".join(_DASAR) + r")\s+puluh(?:\s+(?:" + "|".join(_DASAR) + r"))?)|"
    r"(?:" + "|".join(_DASAR) + r"))"
    rf"(?=\s+(?:{_UNIT_BILANGAN})\b|\b)",
    re.IGNORECASE,
)


def _parse_frasa_bilangan(frasa: str) -> int | None:

    tokens = frasa.casefold().split()
    total = 0
    i = 0
    while i < len(tokens):
        tok = tokens[i]
        if tok == "seratus":
            total += 100
        elif tok == "sepuluh":
            total += 10
        elif tok == "sebelas":
            total += 11
        elif tok in _DASAR:
            val = _DASAR[tok]
            if i + 1 < len(tokens) and tokens[i + 1] in {"belas", "puluh", "ratus"}:
                pengali = tokens[i + 1]
                if pengali == "belas":
                    total += 10 + val
                elif pengali == "puluh":
                    total += val * 10
                else:
                    total += val * 100
                i += 1
            else:
                total += val
        else:
            return None
        i += 1
    return total


def normalisasi_bilangan(teks: str) -> str:

    hasil = _POLA_KURUNG_ANGKA.sub(r"\1", teks)

    def _ganti(match: re.Match) -> str:
        frasa = match.group(0)
        tokens = frasa.casefold().split()

        if len(tokens) == 1:
            sisa = hasil[match.end():].lstrip()
            if not re.match(rf"^(?:{_UNIT_BILANGAN})\b", sisa, re.IGNORECASE):
                return frasa
        angka = _parse_frasa_bilangan(frasa)
        return str(angka) if angka is not None else frasa

    return _POLA_KATA_BILANGAN.sub(_ganti, hasil)


def _ke_float(teks_angka: str) -> float:
    return float(teks_angka.replace(",", "."))


def _cek_rentang(teks: str, nilai: float) -> bool | None:

    dua_sisi = re.search(
        r"(\d+(?:,\d+)?)\s*([<≤]|<=)\s*[A-Za-z]+\s*([<≤]|<=)\s*(\d+(?:,\d+)?)",
        teks,
    )
    if dua_sisi:
        bawah, op1, op2, atas = dua_sisi.groups()
        return _KOMPARATOR[op1](_ke_float(bawah), nilai) and _KOMPARATOR[op2](nilai, _ke_float(atas))

    satu_sisi = re.search(r"\b[A-Za-z]+\s*([≥≤<>]|>=|<=)\s*(\d+(?:,\d+)?)", teks)
    if satu_sisi:
        op, batas = satu_sisi.groups()
        return _KOMPARATOR[op](nilai, _ke_float(batas))
    return None


def evaluasi_tabel(teks: str, pertanyaan: str, tipe_jawaban: str) -> str | None:

    angka_query = re.findall(r"\d+(?:,\d+|\.\d+)?", pertanyaan)
    if angka_query:
        nilai = _ke_float(angka_query[-1].replace(".", ","))
        cocok = _cek_rentang(teks, nilai)
        if cocok is False:
            return None
        if cocok is True:
            if tipe_jawaban == "NILAI_HURUF":
                m = re.search(r"nilai huruf\s+([A-E]\+?)", teks)
                if m:
                    return m.group(1)
            m_adalah = re.search(r"\badalah\s+([^.(]+)", teks)
            if m_adalah:
                return m_adalah.group(1).strip()


    m_huruf_q = re.findall(r"(?<![A-Za-z])([A-E]\+?)(?![A-Za-z0-9+])", pertanyaan)
    if m_huruf_q:
        huruf_target = m_huruf_q[-1]
        pola_baris = re.search(
            rf"(?:Huruf Mutu \(HM\)|nilai huruf)\s+{re.escape(huruf_target)}(?![A-Za-z0-9+])",
            teks,
        )
        if pola_baris:
            m_nilai = re.search(r"(?:Angka Mutu \(AM\)|bobot)\s+(\d+(?:,\d+)?)", teks)
            if m_nilai:
                return m_nilai.group(1)
    return None


def ekstrak_rumus(teks: str) -> str | None:

    m = re.search(r"\b([A-Z]{1,4}\s*=\s*[^.]+)", teks)
    if not m:
        return None
    rumus = re.split(r",\s*dengan\b", m.group(1), maxsplit=1)[0].strip()
    return f"{rumus}."

Jalur simbolik menangani rentang nilai, bilangan yang ditulis dengan huruf, dan rumus.

# 2. Ekstraktor Hybrid

In [2]:
_UNIT_DURASI = frozenset({"menit", "jam", "hari", "minggu", "bulan", "tahun", "semester"})


_UNIT_ANGKA = frozenset({
    "sks", "ipk", "ip", "bobot", "skor", "poin", "kali", "orang", "publikasi", "jenis",
})


_BULAN_ID = frozenset({
    "januari", "februari", "maret", "april", "mei", "juni",
    "juli", "agustus", "september", "oktober", "november", "desember",
})


_NLP = None


def load_udpipe():

    global _NLP
    if _NLP is not None:
        return _NLP
    try:
        _NLP = spacy_udpipe.load("id")
    except (AssertionError, OSError):
        spacy_udpipe.download("id")
        _NLP = spacy_udpipe.load("id")
    return _NLP


def _akar_sederhana(kata: str) -> str:

    w = re.sub(r"[^a-z0-9+]", "", kata.casefold())
    if len(w) <= 4:
        return w
    if w.startswith(("meny", "peny")):
        w = "s" + w[4:]
    elif w.startswith(("men", "pen")) and len(w) > 5 and w[3] in "aeiou":
        w = "t" + w[3:]
    elif w.startswith(("mem", "pem")) and len(w) > 5 and w[3] in "aeiou":
        w = "p" + w[3:]
    elif w.startswith(("meng", "peng")) and len(w) > 6 and w[4] in "aeiou":
        w = w[4:]
    else:
        w = re.sub(r"^(?:meng|peng|mem|pem|men|pen|me|pe|di|ber|ter|ke)", "", w)
    if len(w) > 5:
        w = re.sub(r"(?:kan|an)$", "", w)
    return w


def _akar_kunci(hasil_pertanyaan: dict) -> set[str]:
    kata = hasil_pertanyaan.get("kata_kunci") or []
    return {_akar_sederhana(k) for k in kata if len(k) >= 2}


def _bersihkan_butir(teks: str) -> str:

    return re.sub(r"^\s*(?:\d+|[a-z]{1,2}|[ivx]+)\.\s+", "", teks.strip(), flags=re.IGNORECASE)


def _format_sumber(passage: dict, dokumen: dict | None = None) -> str:
    sumber = passage.get("sumber")
    if not sumber and dokumen and passage.get("dok_id") in dokumen:
        sumber = dokumen[passage["dok_id"]]
    if not sumber:
        return f"{passage.get('passage_id', '')}, hlm. {passage.get('halaman', '-')}"
    bab = passage.get("bab", "")
    subbab = passage.get("subbab", "")
    lokasi = f"BAB {bab} {subbab}".strip() if bab else passage.get("judul_subbab", "")
    bagian = f", {lokasi}" if lokasi else ""
    return f"{sumber['judul']} ({sumber['tahun']}){bagian}, hlm. {passage.get('halaman', '-')}"


def _jarak_terdekat(
    idx_token: int,
    doc,
    akar_kunci: set[str],
    kecualikan: range | None = None,
) -> int:
    cocok = [
        abs(idx_token - t.i)
        for t in doc
        if (kecualikan is None or t.i not in kecualikan)
        and _akar_sederhana(t.text) in akar_kunci
    ]
    return min(cocok) if cocok else 999


def _bonus_dependensi(token, akar_kunci: set[str]) -> float:

    node = token.head
    for langkah in range(2):
        if _akar_sederhana(node.text) in akar_kunci:
            return 1.2 if langkah == 0 else 0.6
        if node.head == node:
            break
        node = node.head
    return 0.0


def _overlap_kalimat(span_or_token, akar_kunci: set[str]) -> int:
    akar_sent = {_akar_sederhana(tok.text) for tok in span_or_token.sent}
    return len(akar_sent & akar_kunci)


def _ekstrak_kuantitatif(doc, tipe: str, akar_kunci: set[str], rank: int) -> tuple[str, float] | None:
    kandidat = []
    for t in doc:
        if t.pos_ != "NUM" or not re.search(r"\d", t.text):
            continue

        if t.i + 1 < len(doc) and doc[t.i + 1].text == ".":
            continue

        teks_num = t.text
        idx_akhir = t.i

        if teks_num.endswith(",") and t.i + 1 < len(doc) and doc[t.i + 1].pos_ == "NUM":
            teks_num = f"{teks_num}{doc[t.i + 1].text}"
            idx_akhir = t.i + 1

        teks_next = doc[idx_akhir + 1].text.casefold() if idx_akhir + 1 < len(doc) else ""
        teks_prev = doc[t.i - 1].text if t.i > 0 else ""

        if tipe == "PERSENTASE":
            if teks_next not in {"%", "persen"}:
                continue
            jawaban = f"{teks_num}%" if teks_next == "%" else f"{teks_num} {doc[idx_akhir + 1].text}"
        elif tipe == "DURASI":
            if teks_next not in _UNIT_DURASI:
                continue
            jawaban = f"{teks_num} {doc[idx_akhir + 1].text}"
        else:
            if teks_next in _UNIT_DURASI and teks_next not in akar_kunci:
                continue
            if teks_next in _UNIT_ANGKA:
                jawaban = f"{teks_num} {doc[idx_akhir + 1].text}"
            elif teks_prev in {"≥", "≤", ">", "<"}:
                jawaban = f"{teks_prev} {teks_num}"
            else:
                jawaban = teks_num

        jarak = _jarak_terdekat(t.i, doc, akar_kunci)
        overlap = _overlap_kalimat(t, akar_kunci)
        skor = (0.8 * overlap) + (2.0 / (jarak + 1.0)) + _bonus_dependensi(t, akar_kunci) + (1.0 / rank)
        kandidat.append((skor, jawaban))

    if not kandidat:
        return None
    kandidat.sort(key=lambda x: -x[0])
    return kandidat[0][1], kandidat[0][0]


def _ekstrak_definisi(doc, akar_kunci: set[str], rank: int) -> tuple[str, float] | None:
    for t in doc:
        if t.dep_ != "cop" or t.text.casefold() not in {"adalah", "merupakan", "ialah", "yaitu"}:
            continue
        subjek = [c for c in t.head.children if "subj" in c.dep_]
        teks_subjek = " ".join(tok.text for s in subjek for tok in s.subtree)
        akar_subjek = {_akar_sederhana(w) for w in teks_subjek.split()}
        overlap = len(akar_subjek & akar_kunci)
        if not overlap and akar_kunci:
            continue
        sent_end = t.sent.end
        potongan = doc[t.i + 1 : sent_end].text.strip()
        if not potongan:
            continue
        jawaban = potongan[0].upper() + potongan[1:]
        skor = overlap + (1.0 / rank)
        return jawaban, skor
    return None


def _ekstrak_jabatan(doc, akar_kunci: set[str], rank: int) -> tuple[str, float] | None:
    kandidat = []
    i = 0
    while i < len(doc):
        t = doc[i]
        if t.text[0].isupper() and t.pos_ in {"PROPN", "NOUN"} and t.dep_ != "punct":
            j = i + 1
            while j < len(doc) and doc[j].text[0].isupper() and doc[j].pos_ in {"PROPN", "NOUN"}:
                j += 1
            span = doc[i:j]
            akar_span = {_akar_sederhana(tok.text) for tok in span}

            if not akar_span.issubset(akar_kunci):
                rentang = range(i, j)
                jarak = min(_jarak_terdekat(k, doc, akar_kunci, rentang) for k in rentang)
                dep_bonus = max(_bonus_dependensi(doc[k], akar_kunci) for k in rentang)
                overlap = _overlap_kalimat(span, akar_kunci)
                skor = (0.8 * overlap) + (2.0 / (jarak + 1.0)) + (1.5 * dep_bonus) + (1.0 / rank)
                kandidat.append((skor, span.text))
            i = j
        else:
            i += 1
    if not kandidat:
        return None
    kandidat.sort(key=lambda x: -x[0])
    return kandidat[0][1], kandidat[0][0]


def _ekstrak_waktu(doc, akar_kunci: set[str], rank: int) -> tuple[str, float] | None:
    for t in doc:
        if t.pos_ != "NUM" or not re.search(r"\d", t.text):
            continue
        next_w = doc[t.i + 1].text.casefold() if t.i + 1 < len(doc) else ""
        if next_w not in _UNIT_DURASI and next_w not in _BULAN_ID:
            continue
        awal = t.i
        if awal >= 2 and doc[awal - 2 : awal].text.casefold() in {"paling lambat", "paling lama", "paling cepat"}:
            awal -= 2
        akhir = t.i + 2
        while akhir < t.sent.end and doc[akhir].text.casefold() not in {"dan", "serta", ",", ";", "."}:
            akhir += 1
        frasa = doc[awal:akhir].text.strip()
        if frasa:
            jawaban = frasa[0].upper() + frasa[1:] + "."
            skor = (2.0 / (_jarak_terdekat(t.i, doc, akar_kunci) + 1.0)) + (1.0 / rank)
            return jawaban, skor
    return None


def _kalimat_terbaik(teks: str, akar_kunci: set[str], rank: int) -> tuple[str, float] | None:
    kalimat_list = [k.strip() for k in re.split(r"(?<=[.!?])\s+", teks) if k.strip()]
    if not kalimat_list:
        return None
    terbaik = max(
        kalimat_list,
        key=lambda k: len({_akar_sederhana(w) for w in k.split()} & akar_kunci),
    )
    overlap = len({_akar_sederhana(w) for w in terbaik.split()} & akar_kunci)
    return terbaik, overlap + (1.0 / rank)


class EkstraktorJawaban:


    def __init__(self, dokumen: dict | None = None, nlp=None):
        self.dokumen = dokumen or {}
        self._nlp = nlp

    @property
    def nlp(self):
        if self._nlp is None:
            self._nlp = load_udpipe()
        return self._nlp

    def ekstrak(self, hasil_pertanyaan: dict, kandidat_passages: list[dict]) -> dict:
        pertanyaan = hasil_pertanyaan.get("pertanyaan", "")
        tipe = hasil_pertanyaan.get("tipe_jawaban", "TIDAK_DIKENALI")

        if hasil_pertanyaan.get("status") != "SIAP_RETRIEVAL" or not kandidat_passages:
            return {
                "pertanyaan": pertanyaan,
                "tipe_jawaban": tipe,
                "jawaban": "",
                "metode": "tidak_tersedia",
                "skor": 0.0,
                "passage_id": "",
                "kutipan": "",
                "sumber": "",
                "status": "TIDAK_DITEMUKAN",
            }

        akar_kunci = _akar_kunci(hasil_pertanyaan)


        for rank_idx, p in enumerate(kandidat_passages, start=1):
            rank = p.get("rank", rank_idx)
            teks = p.get("teks", "")
            if tipe == "RUMUS":
                rumus = ekstrak_rumus(teks)
                if rumus:
                    return self._bangun_hasil(pertanyaan, tipe, rumus, "simbolik", 1.0 / rank, p)
            if p.get("jenis") == "tabel" or tipe == "NILAI_HURUF":
                nilai_tabel = evaluasi_tabel(teks, pertanyaan, tipe)
                if nilai_tabel:
                    return self._bangun_hasil(pertanyaan, tipe, nilai_tabel, "simbolik", 1.0 / rank, p)


        kandidat_linguistik = []
        for rank_idx, p in enumerate(kandidat_passages, start=1):
            rank = p.get("rank", rank_idx)
            teks_bersih = normalisasi_bilangan(_bersihkan_butir(p.get("teks", "")))
            doc = self.nlp(teks_bersih)

            hasil_ling = None
            if tipe in {"ANGKA", "DURASI", "PERSENTASE"}:
                hasil_ling = _ekstrak_kuantitatif(doc, tipe, akar_kunci, rank)
            elif tipe == "DEFINISI":
                hasil_ling = _ekstrak_definisi(doc, akar_kunci, rank)
            elif tipe in {"JABATAN", "ORANG"}:
                hasil_ling = _ekstrak_jabatan(doc, akar_kunci, rank)
            elif tipe == "WAKTU":
                hasil_ling = _ekstrak_waktu(doc, akar_kunci, rank)

            if hasil_ling:
                jawaban, skor = hasil_ling
                kandidat_linguistik.append((skor, jawaban, p))

        if kandidat_linguistik:
            kandidat_linguistik.sort(key=lambda x: -x[0])
            skor_terbaik, jawaban_terbaik, passage_terbaik = kandidat_linguistik[0]
            return self._bangun_hasil(
                pertanyaan, tipe, jawaban_terbaik, "linguistik_udpipe", skor_terbaik, passage_terbaik,
            )


        top_p = kandidat_passages[0]
        fallback = _kalimat_terbaik(_bersihkan_butir(top_p.get("teks", "")), akar_kunci, top_p.get("rank", 1))
        if fallback:
            jawaban_fb, skor_fb = fallback
            return self._bangun_hasil(pertanyaan, tipe, jawaban_fb, "fallback_kalimat", skor_fb, top_p)

        return self._bangun_hasil(pertanyaan, tipe, "", "tidak_tersedia", 0.0, top_p, ditemukan=False)

    def _bangun_hasil(
        self,
        pertanyaan: str,
        tipe: str,
        jawaban: str,
        metode: str,
        skor: float,
        passage: dict,
        ditemukan: bool = True,
    ) -> dict:
        return {
            "pertanyaan": pertanyaan,
            "tipe_jawaban": tipe,
            "jawaban": jawaban,
            "metode": metode,
            "skor": round(float(skor), 4),
            "passage_id": passage.get("passage_id") or passage.get("id", ""),
            "kutipan": passage.get("teks", ""),
            "sumber": _format_sumber(passage, self.dokumen),
            "status": "DITEMUKAN" if ditemukan else "TIDAK_DITEMUKAN",
        }

Ekstraktor mencoba aturan simbolik, kemudian memilih kandidat berdasarkan struktur kalimat dan kedekatan kata. Jika tidak ada kecocokan khusus, sistem memakai kalimat yang paling relevan.

# 3. Jawaban Sistem

In [3]:
with open("../Anggota3_Retrieval/output/hasil_pencarian.json", encoding="utf-8") as file:
    hasil_pencarian = json.load(file)
baris_pencarian = hasil_pencarian["hasil_per_pertanyaan"]

ekstraktor = EkstraktorJawaban()
hasil_ekstraksi = [
    {
        "id_pertanyaan": soal["id_pertanyaan"],
        "jawaban_rujukan": soal["jawaban"],
        **ekstraktor.ekstrak(soal["query"], soal["kandidat_top5"]),
    }
    for soal in baris_pencarian
]

pd.DataFrame(hasil_ekstraksi)[
    ["id_pertanyaan", "jawaban", "jawaban_rujukan", "metode", "passage_id"]
]

,id_pertanyaan,jawaban,jawaban_rujukan,metode,passage_id
0,A2-001,144 sks,144 sks,linguistik_udpipe,pa2020-II-I-7
1,A2-002,75 menit,75 menit,linguistik_udpipe,pa2020-VI-D-10
2,A2-003,Format skripsi disusun menurut dokumen Panduan...,Karya ilmiah mahasiswa pendidikan program sarj...,fallback_kalimat,pa2020-VI-A-14
3,A2-004,"Menurut pedoman penilaian PTIIK, nilai angka (...",B+,fallback_kalimat,penilaian-konversi-1
4,A3-001,14 semester,14 semester sejak pertama kali terdaftar sebag...,linguistik_udpipe,pa2020-II-I-8
5,A3-002,Paling lambat 1 bulan setelah berakhirnya masa...,Paling lambat 1 bulan setelah masa daftar ulan...,linguistik_udpipe,pa2020-III-G-2
6,A3-003,Keberhasilan studi mahasiswa dinyatakan dengan...,IP = Σ (K_i × Na_i) / Σ K_i.,fallback_kalimat,pa2020-II-H-1
7,A4-001,80%,80%,linguistik_udpipe,pa2020-III-K-1
8,A4-002,15 menit,15 menit,linguistik_udpipe,tatatertib-X-S1-3
9,A4-003,"3,5","3,5",simbolik,pa2020-bobot-2


Kolom jawaban menunjukkan teks yang dipilih sistem. Metode dan ID passage memperlihatkan bagaimana jawaban diambil dan dari mana sumbernya.

# 4. Simpan Jawaban

In [4]:
Path("output").mkdir(exist_ok=True)
with open("output/hasil_ekstraksi.json", "w", encoding="utf-8") as file:
    json.dump(hasil_ekstraksi, file, ensure_ascii=False, indent=2)
    file.write("\n")

Jawaban sistem, metode, dan sumber diekspor bersama ID soal untuk penilaian akhir.